In [10]:
%load_ext memory_profiler


The memory_profiler extension is already loaded. To reload it, use:
  %reload_ext memory_profiler


In [11]:
import time

from sdv.relational import HMA1
from sdv import Metadata
import warnings
warnings.filterwarnings('ignore')
from tutorials.relational_data.test_datas.build_meta import build_sdv_metadata_from_origin_tables
from test_20_tables_sdv import fetch_data_from_sqlite_filter, XArgs

TEMP_LOGGER = open("./test_datas/timelog.txt", 'a+')


In [12]:
table_level = '14t'
x_arg = XArgs.tables_14
db_level = '1k'
add_level = '5'
target_name = f'{db_level}{table_level}{add_level}a'
import pickle
with open(f"./test_datas/{target_name}.pkl", 'rb') as f:
    train_data = pickle.load(f)
# train_data

In [13]:
metadata, otables = fetch_data_from_sqlite_filter(x_arg=x_arg,path=f"./test_datas/{db_level}.db")
metadata = build_sdv_metadata_from_origin_tables(train_data, metadata, otables)
metadata = Metadata(metadata)
metadata.to_json(f"./test_datas/{target_name}.json")
metadata

Metadata
  root_path: .
  tables: ['BookLoan', 'Book', 'Library', 'Student', 'Enrollment', 'Submission', 'Course', 'CourseTextbook', 'Textbook', 'Schedule', 'Professor', 'ProjectMember', 'ResearchProject', 'ResearchGroup']
  relationships:
    BookLoan.book_id -> Book.book_id
    BookLoan.student_id -> Student.student_id
    Book.library_id -> Library.library_id
    Enrollment.course_id -> Course.course_id
    Enrollment.student_id -> Student.student_id
    Submission.student_id -> Student.student_id
    CourseTextbook.course_id -> Course.course_id
    CourseTextbook.textbook_id -> Textbook.textbook_id
    Schedule.course_id -> Course.course_id
    Schedule.professor_id -> Professor.professor_id
    ProjectMember.professor_id -> Professor.professor_id
    ProjectMember.project_id -> ResearchProject.project_id
    ResearchProject.group_id -> ResearchGroup.group_id

In [14]:
model = HMA1(metadata)    

In [15]:
st = time.time()
%memit model.fit(train_data)
ed = time.time()
print(f"Time to fit HMA1 model: {ed - st}")
TEMP_LOGGER.write(f"\n{target_name}, train {ed - st}s, ")
TEMP_LOGGER.flush()

Fitting HMA1: 0it [00:00, ?it/s]

Modeling Library
Modeling Book
Modeling BookLoan


Book:BookLoan:   0%|          | 0/1 [00:00<?, ?it/s]

BookLoan Unique:   0%|          | 0/1000 [00:00<?, ?it/s]

Library:Book:   0%|          | 0/1 [00:00<?, ?it/s]

...

ResearchProject:ProjectMember:   0%|          | 0/1 [00:00<?, ?it/s]

ProjectMember Unique:   0%|          | 0/1000 [00:00<?, ?it/s]

ResearchGroup:ResearchProject:   0%|          | 0/1 [00:00<?, ?it/s]

ResearchProject Unique:   0%|          | 0/1000 [00:00<?, ?it/s]

peak memory: 570.38 MiB, increment: 148.33 MiB
Time to fit HMA1 model: 144.79943203926086


In [16]:
def gen_data(num_sample = 10000):
    st = time.time()
    new_data = model.sample(num_rows=num_sample)
    ed = time.time()
    print(f"Time to sample {num_sample} model: {ed - st}")
    TEMP_LOGGER.write(f"{num_sample} sample {ed - st}s, ")
    TEMP_LOGGER.flush()
    return new_data
# new_data

In [17]:
from sdv.evaluation import evaluate
def evaluate_data(num_sample ,synthetic_data, real_data):
    #synthetic_data = new_data
    #real_data = train_data
    st = time.time()
    evadata = evaluate(synthetic_data, real_data,metadata=metadata, aggregate=False)
    ed = time.time()
    print(f"Time to evaluate synthetic data: {ed - st}")
    print(evadata.normalized_score.mean())
    ks, cs = float(evadata[evadata['metric']=='KSComplement']["raw_score"].iloc[0]), float(evadata[evadata['metric']=='CSTest']["raw_score"].iloc[0])
    TEMP_LOGGER.write(f"{num_sample} eva: {ed - st}s, ks: {ks}, cs: {cs}, ")
    TEMP_LOGGER.flush()

In [18]:
%memit data_1k = gen_data(1000)
# evaluate_data(1000,synthetic_data=data_1k,real_data=train_data)
%memit data_10k = gen_data(10000)
# evaluate_data(10000,synthetic_data=data_10k,real_data=train_data)

  0%|          | 0/14 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

Library Childing:   0%|          | 0/1 [00:00<?, ?it/s]

Book:   0%|          | 0/1000 [00:00<?, ?it/s]

Book Childing:   0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/10000 [00:00<?, ?it/s]

Student Childing:   0%|          | 0/3 [00:00<?, ?it/s]

Enrollment:   0%|          | 0/10000 [00:00<?, ?it/s]

Submission:   0%|          | 0/10000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
model.save(path=f"./test_datas/savemodel_{target_name}.pkl")